# Deepfake Detection - Train All Strategies (c23)
Make sure to:
- Set **Accelerator → P100** in notebook settings
- Add dataset `katroue/ff-c23-frames-1` via **+ Add Data**

In [ ]:
# Step 1: Clone repo
!git clone https://github.com/katherinedemers/deepfake_project_comp6341.git
%cd deepfake_project_comp6341

In [ ]:
# Step 2: Install dependencies
!pip install timm scikit-learn grad-cam pyyaml -q

In [ ]:
# Step 3: Map Kaggle dataset structure to what the code expects
import os

KAGGLE_DATA = '/kaggle/input/ff-c23-frames-1'
DATA_ROOT   = '/kaggle/working/deepfake_project_comp6341/data'

dirs = [
    f'{DATA_ROOT}/original_sequences/youtube/c23',
    f'{DATA_ROOT}/manipulated_sequences/Deepfakes/c23',
    f'{DATA_ROOT}/manipulated_sequences/Face2Face/c23',
    f'{DATA_ROOT}/manipulated_sequences/FaceSwap/c23',
    f'{DATA_ROOT}/manipulated_sequences/NeuralTextures/c23',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

# Symlink each sequence's images folder
links = {
    f'{DATA_ROOT}/original_sequences/youtube/c23/images':             f'{KAGGLE_DATA}/original',
    f'{DATA_ROOT}/manipulated_sequences/Deepfakes/c23/images':        f'{KAGGLE_DATA}/Deepfakes',
    f'{DATA_ROOT}/manipulated_sequences/Face2Face/c23/images':        f'{KAGGLE_DATA}/Face2Face',
    f'{DATA_ROOT}/manipulated_sequences/FaceSwap/c23/images':         f'{KAGGLE_DATA}/FaceSwap',
    f'{DATA_ROOT}/manipulated_sequences/NeuralTextures/c23/images':   f'{KAGGLE_DATA}/NeuralTextures',
    f'{DATA_ROOT}/splits':                                             f'{KAGGLE_DATA}/splits',
}
for link, target in links.items():
    if not os.path.exists(link):
        os.symlink(target, link)

# Verify
for link in links:
    count = len(os.listdir(link))
    print(f'{link.split("/data/")[1]}: {count} entries')

In [ ]:
# Step 4: Update data_root in all c23 configs to point to Kaggle data
import glob, re

DATA_ROOT = '/kaggle/working/deepfake_project_comp6341/data/'
for cfg in glob.glob('configs/c23/*.yaml'):
    with open(cfg) as f:
        content = f.read()
    content = re.sub(r'data_root:.*', f'data_root: "{DATA_ROOT}"', content)
    # Update save and log dirs to /kaggle/working for output
    content = re.sub(r'save_dir: "results/', 'save_dir: "/kaggle/working/results/', content)
    content = re.sub(r'log_dir: "results/', 'log_dir: "/kaggle/working/results/', content)
    with open(cfg, 'w') as f:
        f.write(content)
    print(f'Updated {cfg}')

In [ ]:
# Step 5: Run all 6 strategies
!bash scripts/train_all_strategies_c23.sh

In [ ]:
# Step 6: Copy results to Kaggle output so they can be downloaded
import shutil
shutil.copytree('/kaggle/working/results', '/kaggle/output/results', dirs_exist_ok=True)
print('Results saved to /kaggle/output/results')
!find /kaggle/output/results/models -name 'best_model.pth'